# Particle Data: Load Selected Variables and Spatial Ranges

This notebook provides a comprehensive guide to selective particle data loading and spatial filtering in Mera.jl. You'll learn advanced techniques for efficiently loading only the particle data you need from large N-body simulations.

## Learning Objectives

- Master selective particle variable loading for memory optimization
- Apply spatial filtering and region selection techniques for particle populations
- Work with different coordinate systems and units for particle analysis
- Understand center-relative coordinate systems for particle distributions
- Optimize particle data loading for large simulations

## Quick Reference: Particle Data Selection Functions

This section provides a comprehensive reference of Mera.jl functions for selective particle data loading and spatial filtering.

### Variable Selection
```julia
# Load all variables (default behavior)
particles = getparticles(info)

# Select specific variables by name
particles = getparticles(info, vars=[:mass, :vx, :vy])     # Mass and velocities
particles = getparticles(info, vars=[:var4, :var1, :var2]) # Using variable numbers

# Select variables without keyword (order matters: info, variables)
particles = getparticles(info, [:mass, :birth])           # Multiple variables
particles = getparticles(info, :vx)                       # Single variable

# Common particle variable names and numbers (RAMSES 2018+)
# :vx, :vy, :vz     → Velocity components
# :mass             → Particle mass
# :family           → Particle family identifier
# :tag              → Particle tag
# :birth            → Birth time/redshift
# :metals           → Metallicity
# :var9, :var10...  → Additional variables

# RAMSES 2017 and earlier
# :var1, :var2, :var3 → vx, vy, vz
# :var4             → mass
# :var5             → birth
# :var6, :var7...   → Additional variables
```

### Spatial Range Selection
```julia
# RAMSES standard notation (domain: [0:1]³)
particles = getparticles(info, xrange=[0.2, 0.8],        # X-range filter
                              yrange=[0.2, 0.8],        # Y-range filter  
                              zrange=[0.4, 0.6])        # Z-range filter

# Center-relative coordinates (RAMSES units)
particles = getparticles(info, xrange=[-0.3, 0.3],       # Relative to center
                              yrange=[-0.3, 0.3],
                              zrange=[-0.1, 0.1],
                              center=[0.5, 0.5, 0.5])

# Physical units (e.g., kpc)
particles = getparticles(info, xrange=[2., 22.],          # Physical coordinates
                              yrange=[2., 22.],
                              zrange=[22., 26.],
                              range_unit=:kpc)

# Center-relative with physical units
particles = getparticles(info, xrange=[-16., 16.],        # Relative to center in kpc
                              yrange=[-16., 16.],
                              zrange=[-2., 2.],
                              center=[24., 24., 24.],
                              range_unit=:kpc)

# Box center shortcuts
particles = getparticles(info, center=[:boxcenter])      # All dimensions centered
particles = getparticles(info, center=[:bc])             # Short form
particles = getparticles(info, center=[:bc, 24., :bc])   # Mixed: center x,z; fixed y
```

### PerformanceOptimization
```julia
# Combined optimizations for particles
particles = getparticles(info, [:mass, :vx, :vy, :vz],   # Select variables
                              xrange=[-10., 10.],        # Spatial range
                              yrange=[-10., 10.],
                              zrange=[-2., 2.],
                              center=[:bc],              # Box center
                              range_unit=:kpc)           # Physical units
```

### Available Physical Units
```julia
# Check available units in simulation
viewfields(info.scale)

# Common length units
:m, :km, :cm, :mm, :μm, :Mpc, :kpc, :pc, :ly, :au, :Rsun
```

## Getting Started: Simulation Setup

Before exploring particle data selection techniques, let's load our simulation and examine its properties. This establishes the foundation for all subsequent particle data loading operations.

In [1]:
using Mera
info = getinfo(300, "/Volumes/FASTStorage/Simulations/Mera-Tests/RAMSES/mw_L10");

[Mera]: 2026-06-01T14:16:24.675



Code: RAMSES
output [300] summary:


mtime: 2023-04-09T05:34:09
ctime: 2025-06-21T18:31:24.020
simulation time: 445.89 [Myr]
boxlen: 48.0 [kpc]
ncpu: 640
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 10 --> cellsize(s): 750.0 [pc] - 46.88 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  

7  --> (:rho, :vx, :vy, :vz, :p, :var6, :var7)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :pressure, :scalar_00, :scalar_01)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Nstars:   5.445150e+05 
particle-variables: 

7  --> (:vx, :vy, :vz, :mass, :family, :tag, :birth)
particle-descriptor: (:position_x, :position_y, :position_z, :velocity_x, :velocity_y, :velocity_z, :mass, :identity, :levelp, :family, :tag, :birth_time)
-------------------------------------------------------
rt:            false
clumps:           false
-------------------------------------------------------
namelist-file: ("&COOLING_PARAMS", "&SF_PARAMS", "&AMR_PARAMS", "&BOUNDARY_PARAMS", "&OUTPUT_PARAMS", "&POISSON_PARAMS", "&RUN_PARAMS", "&FEEDBACK_PARAMS", "&HYDRO_PARAMS", "&INIT_PARAMS", "&REFINE_PARAMS")
-------------------------------------------------------
timer-file:       true
compilation-file: false
makefile:         true
patchfile:        true



## Variable Selection Techniques

Understanding how to selectively load particle variables is crucial for efficient memory usage and faster analysis. Mera provides flexible approaches to particle variable selection, from loading everything to precise property targeting.

### Understanding Particle Variable References

Mera provides flexible ways to reference particle properties with support for different RAMSES versions. Understanding these reference methods enables precise control over particle data loading.

**RAMSES 2018 and Later Variable References:**

| Variable | Symbol Format | Number Format | Description |
|----------|---------------|---------------|-------------|
| X-Velocity | `:vx` | `:var1` | Velocity component in x-direction |
| Y-Velocity | `:vy` | `:var2` | Velocity component in y-direction |
| Z-Velocity | `:vz` | `:var3` | Velocity component in z-direction |
| Mass | `:mass` | `:var4` | Particle mass |
| Family | `:family` | `:var5` | Particle family identifier |
| Tag | `:tag` | `:var6` | Particle tag |
| Birth Time | `:birth` | `:var7` | Birth time/redshift |
| Metallicity | `:metals` | `:var8` | Metal content |
| Additional | - | `:var9`, `:var10`, ... | Extended properties |

**RAMSES 2017 and Earlier:**

| Variable | Number Format | Description |
|----------|---------------|-------------|
| X-Velocity | `:var1` | Velocity component in x-direction |
| Y-Velocity | `:var2` | Velocity component in y-direction |
| Z-Velocity | `:var3` | Velocity component in z-direction |
| Mass | `:var4` | Particle mass |
| Birth Time | `:var5` | Birth time/redshift |
| Additional | `:var6`, `:var7`, ... | Extended properties |

**Always Available (Position and Identification):**
- Position data: `:level`, `:x`, `:y`, `:z`
- Identifiers: `:id`, `:cpu` (or `:varn1`)

**Key Features:**
- Version-dependent variable naming conventions
- Both symbolic and numeric formats supported  
- Future support for descriptor file variable names
- Consistent API across RAMSES versions

### Loading All Variables (Default Behavior)

The simplest approach is to load all available particle variables. This is the default behavior when no specific variables are requested.

In [2]:
particles = getparticles(info);

[Mera]: Get particle data: 2026-06-01T14:16:28.733



Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 



domain:


xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing


Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

38.428720474243164 MB
-------------------------------------------------------



In [3]:
particles.data

Table with 544515 rows, 12 columns:
Columns:
#   colname  type
────────────────────
1   level    Int32
2   x        Float64
3   y        Float64
4   z        Float64
5   id       Int32
6   family   Int8
7   tag      Int8
8   vx       Float64
9   vy       Float64
10  vz       Float64
11  mass     Float64
12  birth    Float64

### Selecting Multiple Variables

Mera provides multiple ways to select specific particle properties. You can use keyword arguments or positional arguments with flexible syntax.

In [4]:
particles_a = getparticles(info, vars=[:mass, :birth]); 

[Mera]: Get particle data: 2026-06-01T14:16:32.778

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(4, 7) = (:mass, :birth) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

25.965506553649902 MB
-------------------------------------------------------



**Alternative:** Use variable numbers instead of symbolic names. This approach provides identical functionality with numeric references:

In [5]:
particles_a = getparticles(info, vars=[:var4, :var7]); 

[Mera]: Get particle data: 2026-06-01T14:16:33.074

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(4, 7) = (:mass, :birth) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

25.965506553649902 MB
-------------------------------------------------------



**Keyword-free syntax:** When following the specific order (InfoType object, then variables), keyword arguments are optional:

In [6]:
particles_a = getparticles(info, [:mass, :birth]); 

[Mera]: Get particle data: 2026-06-01T14:16:33.236

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(4, 7) = (:mass, :birth) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

25.965506553649902 MB
-------------------------------------------------------



In [7]:
particles_a.data

Table with 544515 rows, 9 columns:
level  x        y        z        id      family  tag  mass        birth
──────────────────────────────────────────────────────────────────────────
9      9.17918  22.4404  24.0107  128710  2       0    8.00221e-7  8.86726
9      9.23642  21.5559  24.0144  126838  2       0    8.00221e-7  8.71495
9      9.35638  20.7472  24.0475  114721  2       0    8.00221e-7  7.91459
9      9.39529  21.1854  24.0155  113513  2       0    8.00221e-7  7.85302
9      9.42686  20.9697  24.0162  120213  2       0    8.00221e-7  8.2184
9      9.42691  22.2181  24.0137  125689  2       0    8.00221e-7  8.6199
9      9.48834  22.0913  24.0137  126716  2       0    8.00221e-7  8.70493
9      9.5262   20.652   24.0179  115550  2       0    8.00221e-7  7.96008
9      9.60376  21.2814  24.0155  116996  2       0    8.00221e-7  8.03346
9      9.6162   20.6243  24.0506  125003  2       0    8.00221e-7  8.56482
9      9.62155  20.6248  24.0173  112096  2       0    8.00221e-7  7.

### Selecting Single Variables

For single variable selection, arrays and keywords are unnecessary. Maintain the order: InfoType object, then variable symbol:

In [8]:
particles_c = getparticles(info, :vx ); 

[Mera]: Get particle data: 2026-06-01T14:16:33.550

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1,) = (:vx,) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

21.81110191345215 MB
-------------------------------------------------------



In [9]:
particles_c.data

Table with 544515 rows, 8 columns:
level  x        y        z        id      family  tag  vx
─────────────────────────────────────────────────────────────────
9      9.17918  22.4404  24.0107  128710  2       0    0.670852
9      9.23642  21.5559  24.0144  126838  2       0    0.810008
9      9.35638  20.7472  24.0475  114721  2       0    0.93776
9      9.39529  21.1854  24.0155  113513  2       0    0.870351
9      9.42686  20.9697  24.0162  120213  2       0    0.899373
9      9.42691  22.2181  24.0137  125689  2       0    0.717235
9      9.48834  22.0913  24.0137  126716  2       0    0.739564
9      9.5262   20.652   24.0179  115550  2       0    0.946747
9      9.60376  21.2814  24.0155  116996  2       0    0.893236
9      9.6162   20.6243  24.0506  125003  2       0    0.996445
9      9.62155  20.6248  24.0173  112096  2       0    0.960817
9      9.62252  24.4396  24.0206  136641  2       0    0.239579
⋮
10     37.7913  25.6793  24.018   141792  2       0    -0.466362
10     

## Spatial Range Selection Techniques

Spatial filtering is essential for focusing analysis on specific particle populations within regions of interest. Mera offers multiple coordinate systems and reference methods to accommodate different particle analysis needs.

**Available Coordinate Systems:**
- **RAMSES Standard:** Normalized domain [0:1]³ 
- **Center-Relative:** Coordinates relative to specified points
- **Physical Units:** Real astronomical units (kpc, pc, etc.)
- **Box-Centered:** Convenient shortcuts for simulation center

This flexibility allows precise particle population selection for targeted analysis while optimizing memory usage and computational efficiency.

### RAMSES Standard Coordinate System

The RAMSES standard provides a normalized coordinate system that simplifies numerical calculations and ensures consistency across different simulation scales for particle analysis.

**Coordinate System Properties:**
- **Domain Range:** [0:1]³ in all dimensions
- **Origin:** Located at [0., 0., 0.]
- **Benefits:** Scale-independent, numerically stable
- **Usage:** Ideal for relative positioning and particle comparisons

**Particle-Specific Advantage:** This notation is particularly effective for comparing particle distributions with grid-based hydro data, enabling multi-physics analysis.

In [10]:
particles = getparticles(  info, 
                            xrange=[0.2,0.8], 
                            yrange=[0.2,0.8], 
                            zrange=[0.4,0.6]); 

[Mera]: Get particle data: 2026-06-01T14:16:34.177



Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 

domain:
xmin::xmax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
ymin::ymax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
zmin::zmax: 0.4 :: 0.6  	==> 19.2 [kpc] :: 28.8 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing


Combining results from 4 thread(s)...
Found 5.444850e+05 particles
Memory used for data table :

38.42660331726074 MB
-------------------------------------------------------



**Range Verification:** The loaded particle data ranges are stored in the `ranges` field using RAMSES standard notation (domain: [0:1]³):

In [11]:
particles.ranges

6-element Vector{Float64}:
 0.2
 0.8
 0.2
 0.8
 0.4
 0.6

### Center-Relative Coordinate Selection

Define spatial ranges relative to a specified center point. This approach is particularly useful for analyzing particle populations around specific features, galaxies, or objects of interest:

In [12]:
particles = getparticles(  info, 
                            xrange=[-0.3, 0.3], 
                            yrange=[-0.3, 0.3], 
                            zrange=[-0.1, 0.1], 
                            center=[0.5, 0.5, 0.5]);

[Mera]: Get particle data: 2026-06-01T14:16:35.553

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
ymin::ymax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
zmin::zmax: 0.4 :: 0.6  	==> 19.2 [kpc] :: 28.8 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 5.444850e+05 particles
Memory used for data table :

38.42660331726074 MB
-------------------------------------------------------



### Physical Unit Coordinate System

Working with physical units provides intuitive scale references for astronomical particle analysis. This system automatically handles unit conversions and maintains physical meaning for particle distributions.

**Key Advantages:**
- **Intuitive Scaling:** Use familiar astronomical units (kpc, pc, Mpc)
- **Automatic Conversion:** Mera handles unit transformations internally
- **Reference Point:** Coordinates measured from box corner [0., 0., 0.]
- **Flexibility:** Mix different units as needed for particle analysis

The following example demonstrates kiloparsec (kpc) coordinate selection for particle populations:

In [13]:
particles = getparticles(  info, 
                            xrange=[2.,22.], 
                            yrange=[2.,22.], 
                            zrange=[22.,26.], 
                            range_unit=:kpc); 

[Mera]: Get particle data: 2026-06-01T14:16:36.615

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 

domain:
xmin::xmax: 0.0416667 :: 0.4583333  	==> 2.0 [kpc] :: 22.0 [kpc]
ymin::ymax: 0.0416667 :: 0.4583333  	==> 2.0 [kpc] :: 22.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 3.091600e+04 particles
Memory used for data table :

2.183063507080078 MB
-------------------------------------------------------



**Available Physical Units:** The `range_unit` keyword accepts various length units defined in the simulation's `scale` field:

In [14]:
viewfields(info.scale)  # or e.g.: gas.info.scale


[Mera]: Fields to scale from user/code units to selected units
Mpc	= 0.0010000000000006482
kpc	= 1.0000000000006481
pc	= 1000.0000000006482
mpc	= 1.0000000000006482e6
ly	= 3261.5637769461323
Au	= 2.0626480623310105e23
km	= 3.0856775812820004e16
m	= 3.085677581282e19
cm	= 3.085677581282e21
mm	= 3.085677581282e22
μm	= 3.085677581282e25
Mpc3	= 1.0000000000019446e-9
kpc3	= 1.0000000000019444
pc3	= 1.0000000000019448e9
mpc3	= 1.0000000000019446e18
ly3	= 3.469585750743794e10
Au3	= 8.775571306099254e69
km3	= 2.9379989454983075e49
m3	= 2.9379989454983063e58
cm3	= 2.9379989454983065e64
mm3	= 2.937998945498306e67
μm3	= 2.937998945498306e76
Msol_pc3	= 0.9997234790001649
Msun_pc3	= 0.9997234790001649
g_cm3	= 6.76838218451376e-23
Msol_pc2	= 999.7234790008131
Msun_pc2	= 999.7234790008131
g_cm2	= 0.20885045168302602
Gyr	= 0.014910986463557083
Myr	= 14.910986463557084
yr	= 1.4910986463557083e7
s	= 4.70554946422349e14
ms	= 4.70554946422349e17
Msol	= 9.99723479002109e8
Msun	= 9.99723479002109e8
Mearth	

**Center-Relative with Physical Units:** Combine center-relative positioning with physical unit specifications for precise particle population analysis:

In [15]:
particles = getparticles(  info,
                            xrange=[-16.,16.],
                            yrange=[-16.,16.],
                            zrange=[-2.,2.],
                            center=[24.,24.,24.],   # box centre of the 48 kpc box (or use center=[:bc])
                            range_unit=:kpc);

[Mera]: Get particle data: 2026-06-01T14:16:36.774
Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth)
center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]
domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]
Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 1.072360e+05 particles
Memory used for data table :8.18 MB
-------------------------------------------------------


### Box Center Coordinate Shortcuts

Mera provides convenient shortcuts for box-centered coordinate systems, simplifying particle analysis focused on the simulation center.

**Available Shortcuts:**
- `:bc` or `:boxcenter` - Center coordinate for all dimensions  
- Can be applied to individual dimensions selectively
- Combines seamlessly with physical units and range specifications
- Ideal for symmetric particle analysis around simulation center

**Particle-Specific Benefits:**
- Perfect for galaxy-centered particle analysis
- Eliminates manual center calculation for particle distributions
- Ensures precise geometric centering of particle selections
- Simplifies symmetric region definitions for particle populations
- Reduces coordinate specification errors in particle filtering

In [16]:
particles = getparticles(  info, 
                            xrange=[-16.,16.], 
                            yrange=[-16.,16.], 
                            zrange=[-2.,2.], 
                            center=[:boxcenter], 
                            range_unit=:kpc); 

[Mera]: Get particle data: 2026-06-01T14:16:37.021



Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing


Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

38.428720474243164 MB
-------------------------------------------------------



In [17]:
particles = getparticles(  info, 
                            xrange=[-16.,16.], 
                            yrange=[-16.,16.], 
                            zrange=[-2.,2.], 
                            center=[:bc], 
                            range_unit=:kpc); 

[Mera]: Get particle data: 2026-06-01T14:16:38.145

Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 5.445150e+05 particles
Memory used for data table :

38.428720474243164 MB
-------------------------------------------------------



**Selective Dimension Centering:** Apply box center notation to specific dimensions while maintaining explicit coordinates for others. This example centers x and z dimensions while fixing y at 50 kpc:

In [18]:
particles = getparticles(  info, 
                            xrange=[-16.,16.], 
                            yrange=[-16.,16.], 
                            zrange=[-2.,2.], 
                            center=[:bc, 50., :bc], 
                            range_unit=:kpc); 

[Mera]: Get particle data: 2026-06-01T14:16:39.390



Using threaded processing with 4 threads
Key vars=(:level, :x, :y, :z, :id, :family, :tag)
Using var(s)=(1, 2, 3, 4, 7) = (:vx, :vy, :vz, :mass, :birth) 

center: [0.5, 1.0416667, 0.5] ==> [24.0 [kpc] :: 50.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.7083333 :: 1.0  	==> 34.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

Processing 640 CPU files using 4 threads
Mode: Threaded processing
Combining results from 4 thread(s)...
Found 2.078000e+03 particles
Memory used for data table :

151.4609375 KB
-------------------------------------------------------



## Summary

This notebook demonstrated comprehensive particle data selection techniques in Mera.jl, covering both variable selection and spatial filtering strategies for N-body particle data. Key concepts covered include:

### Variable Selection Mastery
- **Flexible Reference Systems:** Using both symbolic (`:mass`) and numeric (`:var4`) variable references
- **Version Compatibility:** Handling RAMSES 2017/2018+ variable naming differences
- **Selective Loading:** Choosing specific particle properties to optimize memory usage  
- **Syntax Variations:** Keyword and positional argument approaches for different coding styles
- **Single vs. Multiple Variables:** Appropriate syntax for different selection scenarios

### Spatial Filtering Expertise  
- **Coordinate Systems:** RAMSES standard, physical units, center-relative, and box-centered approaches
- **Particle-Specific Applications:** Galaxy-centered analysis and particle population filtering
- **Performance Optimization:** Using spatial bounds and targeted particle selections
- **Unit Flexibility:** Working with various astronomical length scales for particle analysis
- **Center Definitions:** Absolute positioning and relative coordinate systems for particle distributions

### Advanced Particle Techniques
- **Combined Selection:** Integrating variable selection with spatial filtering for particles
- **Memory Management:** Balancing analysis needs with computational resources for large N-body datasets
- **Coordinate Shortcuts:** Using box center notation for simplified particle positioning
- **Quality Assurance:** Verifying loaded particle data ranges and population counts
- **Multi-Physics Integration:** Preparing particle data for combined hydro-particle analysis